In [1]:
import cv2
import glob
from ultralytics import YOLO

# 1. Cargar tu modelo entrenado
ruta_pesos = '../models/entrenamiento_script_estable/weights/best.pt'
model = YOLO(ruta_pesos)

# Nombres de tus clases para que el reporte sea legible
nombres_clases = {0: 'Vehículo', 1: 'Peatón', 2: 'Dos_Ruedas', 3: 'Señales'}

# 2. Tomar una sola imagen de prueba para analizar
imagen_prueba = glob.glob('../data/enhanced/images/test/*.jpg')[0]

print(f"Analizando imagen: {imagen_prueba}\n")
print("-" * 50)

# 3. Hacer la inferencia (sin dibujar, solo queremos los datos matemáticos)
resultados = model(imagen_prueba, verbose=False)

# 4. Dimensiones de la imagen para calcular proporciones
img_alto, img_ancho = resultados[0].orig_shape
area_total_imagen = img_ancho * img_alto

# 5. Extraer y evaluar cada objeto detectado
for resultado in resultados:
    cajas = resultado.boxes
    
    if len(cajas) == 0:
        print("No se detectaron objetos de riesgo en esta imagen.")
        continue

    for caja in cajas:
        # Extraer datos básicos
        clase_id = int(caja.cls[0])
        confianza = float(caja.conf[0])
        nombre_objeto = nombres_clases[clase_id]
        
        # Omitir detecciones con baja seguridad (menos del 40%)
        if confianza < 0.4:
            continue
            
        # Extraer coordenadas (x1, y1, x2, y2)
        x1, y1, x2, y2 = caja.xyxy[0].tolist()
        
        # Calcular el Área del objeto (Ancho * Alto)
        ancho_obj = x2 - x1
        alto_obj = y2 - y1
        area_obj = ancho_obj * alto_obj
        
        # Calcular qué porcentaje de la pantalla ocupa (Proximidad)
        porcentaje_pantalla = (area_obj / area_total_imagen) * 100
        
        # --- LÓGICA DE RIESGO HÍBRIDA ---
        nivel_riesgo = "Bajo"
        alerta = ""
        
        # Regla 1: Vehículos muy cerca (ocupan más del 15% de la pantalla)
        if clase_id == 0 and porcentaje_pantalla > 15.0:
            nivel_riesgo = "ALTO"
            alerta = "¡ALERTA DE COLISIÓN INMINENTE!"
            
        # Regla 2: Peatones o Dos Ruedas moderadamente cerca (son vulnerables, el límite es menor)
        elif clase_id in [1, 2] and porcentaje_pantalla > 5.0:
            nivel_riesgo = "CRÍTICO"
            alerta = "¡FRENADO DE EMERGENCIA - USUARIO VULNERABLE!"
            
        # Regla 3: Precaución general por cercanía media
        elif porcentaje_pantalla > 8.0:
            nivel_riesgo = "MEDIO"
            alerta = "Mantener distancia de seguridad."
            
        # Imprimir el reporte del objeto
        print(f"Objeto: {nombre_objeto} (Seguridad: {confianza*100:.1f}%)")
        print(f"-> Proximidad (Tamaño): Ocupa el {porcentaje_pantalla:.2f}% del campo visual.")
        print(f"-> Riesgo Asignado: {nivel_riesgo} {alerta}")
        print("-" * 50)

Analizando imagen: ../data/enhanced/images/test\cabc30fc-e7726578.jpg

--------------------------------------------------
Objeto: Vehículo (Seguridad: 91.6%)
-> Proximidad (Tamaño): Ocupa el 1.45% del campo visual.
-> Riesgo Asignado: Bajo 
--------------------------------------------------
Objeto: Vehículo (Seguridad: 86.3%)
-> Proximidad (Tamaño): Ocupa el 0.62% del campo visual.
-> Riesgo Asignado: Bajo 
--------------------------------------------------
Objeto: Vehículo (Seguridad: 77.8%)
-> Proximidad (Tamaño): Ocupa el 0.17% del campo visual.
-> Riesgo Asignado: Bajo 
--------------------------------------------------
Objeto: Peatón (Seguridad: 71.0%)
-> Proximidad (Tamaño): Ocupa el 0.59% del campo visual.
-> Riesgo Asignado: Bajo 
--------------------------------------------------
Objeto: Señales (Seguridad: 52.3%)
-> Proximidad (Tamaño): Ocupa el 0.10% del campo visual.
-> Riesgo Asignado: Bajo 
--------------------------------------------------
Objeto: Señales (Seguridad: 45.